<a href="https://colab.research.google.com/github/gcetin74/merhaba/blob/main/depresyon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# Öğrenci Depresyon Veri Analizi

Bu analiz çalışmasında aşağıdaki adımları gerçekleştireceğiz:
1. **Veri Yükleme ve Ön İnceleme**: Veri setini yükleyip genel yapısına bakacağız.
2. **Betimsel İstatistikler**: Değişkenlerin merkezi eğilim ve dağılım ölçülerini inceleyeceğiz.
3. **Grup Karşılaştırmaları**: Belirli gruplar (örneğin cinsiyet veya akademik başarı grupları) arasında depresyon skorları veya ilgili metrikler açısından anlamlı farklar olup olmadığını test edeceğiz.
4. **İlişki Analizi**: Sayısal değişkenler arasındaki korelasyonu inceleyeceğiz.
```

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Veriyi yükleme
df = pd.read_csv('/content/Student_Depression_Dataset.csv')

# İlk 5 satırı ve genel yapıyı görüntüleme
print("Veri Seti Bilgisi:")
display(df.info())
print("\nİlk 5 Satır:")
display(df.head())

Veri Seti Bilgisi:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4464 entries, 0 to 4463
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Gender               4464 non-null   object 
 1   Age                  4464 non-null   int64  
 2   Family_background    4464 non-null   object 
 3   Relationship_status  4464 non-null   object 
 4   Marital_status       4464 non-null   object 
 5   Living_style         4464 non-null   object 
 6   Current_semester     4464 non-null   int64  
 7   CGPA                 4464 non-null   float64
 8   friend_circle        4464 non-null   object 
 9   Health_condition     4464 non-null   object 
 10  Cost_without_study   4464 non-null   int64  
 11  Home_division        4464 non-null   object 
 12  Depression           4464 non-null   object 
dtypes: float64(1), int64(3), object(9)
memory usage: 453.5+ KB


None


İlk 5 Satır:


,Gender,Age,Family_background,Relationship_status,Marital_status,Living_style,Current_semester,CGPA,friend_circle,Health_condition,Cost_without_study,Home_division,Depression
0,Female,22,Middle class,Single,Unmarried,Bachelor,8,3.43,Yes,Critical,10113,Chittagong,Yes
1,Male,19,Middle class,Single,Unmarried,Bachelor,2,3.70,Yes,Abnormal,13815,Chittagong,Yes
2,Male,23,Middle class,Single,Unmarried,Family,8,3.28,No,Abnormal,15450,Dhaka,Yes
3,Male,20,Middle class,Complicated,Unmarried,Family,2,3.77,No,Good,17413,Dhaka,No
4,Female,21,Middle class,Single,Unmarried,Family,1,4.00,No,Abnormal,16693,Dhaka,Yes


```markdown
## 1. Betimsel İstatistikler (Descriptive Statistics)
Sayısal ve kategorik değişkenlerin genel dağılımlarını inceliyoruz.
```

In [4]:
# Sayısal değişkenlerin betimsel istatistikleri
print("Sayısal Değişkenlerin Özeti:")
display(df.describe().T)

# Kategorik değişkenlerin frekansları (Örnek olarak Gender ve Depression sütunları için)
print("\nCinsiyet Dağılımı:")
display(df['Gender'].value_counts())

if 'Depression' in df.columns:
    print("\nDepresyon Durumu Dağılımı:")
    display(df['Depression'].value_counts())

Sayısal Değişkenlerin Özeti:


,count,mean,std,min,25%,50%,75%,max
Age,4464.0,22.619176,2.022980,17.00,21.00,23.00,24.0,28.0
Current_semester,4464.0,5.401210,2.578476,0.00,3.00,5.00,7.0,13.0
CGPA,4464.0,3.412753,0.385605,2.25,3.15,3.49,3.7,4.0
Cost_without_study,4464.0,11242.978271,6640.144019,0.00,6100.00,9500.00,15439.0,32000.0



Cinsiyet Dağılımı:


,count
Gender,
Female,2462
Male,2002



Depresyon Durumu Dağılımı:


,count
Depression,
No,3263
Yes,1201


```markdown
## 2. Grup Karşılaştırması (Group Comparison)
Cinsiyete (Gender) göre Akademik Baskı (Academic Pressure) veya Çalışma Sıklığı gibi puanların farklılaşıp farklılaşmadığını incelemek için parametrik (T-Testi) veya parametrik olmayan (Mann-Whitney U) testler uygulayabiliriz.
```

In [1]:
# Grup karşılaştırma analizi
# Örnek: Cinsiyete (Gender) göre Akademik Baskı (Academic Pressure) karşılaştırması
if 'Gender' in df.columns and 'Academic Pressure' in df.columns:
    genders = df['Gender'].dropna().unique()
    if len(genders) >= 2:
        group1 = df[df['Gender'] == genders[0]]['Academic Pressure'].dropna()
        group2 = df[df['Gender'] == genders[1]]['Academic Pressure'].dropna()

        # Normallik kontrolü (Shapiro-Wilk)
        _, p_norm1 = stats.shapiro(group1.sample(min(len(group1), 5000)))
        _, p_norm2 = stats.shapiro(group2.sample(min(len(group2), 5000)))

        print(f"{genders[0]} grubu normallik p-değeri: {p_norm1:.4f}")
        print(f"{genders[1]} grubu normallik p-değeri: {p_norm2:.4f}")

        # Mann-Whitney U Testi (Genellikle anket verilerinde normallik varsayımı sağlanmadığı için tercih edilir)
        stat, p_val = stats.mannwhitneyu(group1, group2)
        print(f"\nMann-Whitney U Testi Sonucu:")
        print(f"İstatistik: {stat:.4f}, p-değeri: {p_val:.4e}")

        # Görselleştirme
        plt.figure(figsize=(8, 5))
        sns.boxplot(x='Gender', y='Academic Pressure', data=df)
        plt.title('Cinsiyete Göre Akademik Baskı Düzeyi')
        plt.show()

NameError: name 'df' is not defined

```markdown
## 3. İlişki Analizi (Correlation Analysis)
Sayısal değişkenler arasındaki doğrusal ilişkileri tespit etmek amacıyla korelasyon matrisi oluşturuyoruz.
```

In [2]:
# Sayısal sütunları seçme
numeric_cols = df.select_dtypes(include=[np.number]).columns

# Korelasyon matrisi hesaplama (Spearman ordinal veriler için daha uygundur)
corr_matrix = df[numeric_cols].corr(method='spearman')

# Isı haritası (Heatmap) görselleştirmesi
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Sayısal Değişkenler Arası Spearman Korelasyon Matrisi')
plt.show()

NameError: name 'df' is not defined